In [1]:
from data import *

In [2]:
train, test, sub = get_train_test()
train_added = add_extra_data(train)
train_added.rename(columns={'SMILES':'SMILES_raw'}, inplace=True)
train_added['SMILES'] = train_added['SMILES_raw'].apply(replace_all_R_with_C)
train_cleaned = clean_smiles(train_added)
train_filtered = filter_train_data(train_cleaned)

train_csv found at kaggle\input\neurips-open-polymer-prediction-2025\train.csv [OK]
test_csv found at kaggle\input\neurips-open-polymer-prediction-2025\test.csv [OK]
sample_submission found at kaggle\input\neurips-open-polymer-prediction-2025\sample_submission.csv [OK]
tc_smiles found at kaggle\input\tc-smiles\Tc_SMILES.csv [OK]
sed_bigsmiles found at kaggle\input\smiles-extra-data\JCIM_sup_bigsmiles.csv [OK]
sed_tg3 found at kaggle\input\smiles-extra-data\data_tg3.xlsx [OK]
sed_dnst1 found at kaggle\input\smiles-extra-data\data_dnst1.xlsx [OK]
dataset4 found at kaggle\input\neurips-open-polymer-prediction-2025\train_supplement\dataset4.csv [OK]
dataset1 found at kaggle\input\neurips-open-polymer-prediction-2025\train_supplement\dataset1.csv [OK]
dataset2 found at kaggle\input\neurips-open-polymer-prediction-2025\train_supplement\dataset2.csv [OK]
dataset3 found at kaggle\input\neurips-open-polymer-prediction-2025\train_supplement\dataset3.csv [OK]
-------------------------------------

In [3]:
symbols = count_symbols_in_df(train_filtered, "SMILES")

In [4]:
print(symbols)

{'*': 18588, 'C': 251669, 'N': 16027, 'O': 33799, 'F': 6666, 'S': 2059, 'Cl': 564, 'Si': 631, 'Na': 7, 'H': 112, 'P': 323, 'Br': 255, 'Ge': 5, 'Se': 7, 'Sn': 7, 'I': 13, 'Cd': 1, 'B': 2, 'Te': 1, 'Ca': 1}


In [6]:
for z in [2, 10, 18, 1, 29, 7]:  # He, Ne, Ar
    print(z, "grid:", z_to_grid_xy(z))

2 grid: (18.0, 1.0)
10 grid: (18.0, 2.0)
18 grid: (18.0, 3.0)
1 grid: (1.0, 1.0)
29 grid: (11.0, 4.0)
7 grid: (15.0, 2.0)


In [7]:
# language: python
# Notebook helper: 检查 SMILES 是否能计算 make_node_features 所需字段
# 直接运行此单元；如 notebook 已有 train_filtered 会自动使用它
import pandas as pd
from collections import Counter, defaultdict
from rdkit import Chem
from rdkit.Chem import rdPartialCharges
from tqdm import tqdm
import math

# 配置（可修改）
compute_gasteiger = True   # 是否在检查前计算 Gasteiger 电荷
max_examples_print = 8     # 打印示例数
# 确定 SMILES 列表来源（优先使用 train_filtered if present）
try:
    smiles_list = list(train_filtered["SMILES"].astype(str))
    print(f"Using train_filtered with {len(smiles_list)} SMILES")
except Exception:
    # 你也可以直接在这里写 smiles_list = ["CCO","C","[R]C"] 等
    smiles_list = [
        "CCO", "C", "c1ccccc1", "C1CC1", "O=C=O", "C([H])", "invalid_smiles"
    ]
    print("train_filtered not found — using example list")

# 要检查的字段（与 make_node_features 中使用的字段对应）
atom_checks = [
    ("GetMass", lambda a: a.GetMass()),
    ("GetAtomMapNum", lambda a: a.GetAtomMapNum()),
    ("GetIsAromatic", lambda a: a.GetIsAromatic()),
    ("GetFormalCharge", lambda a: a.GetFormalCharge()),
    ("GetAtomicNum", lambda a: a.GetAtomicNum()),
    ("GetChiralTag", lambda a: a.GetChiralTag()),
    ("GetHybridization", lambda a: a.GetHybridization()),
    ("GetDegree", lambda a: a.GetDegree()),
    ("GetTotalNumHs", lambda a: a.GetTotalNumHs()),
    ("IsInRing", lambda a: a.IsInRing()),
    ("HasGasteigerProp", lambda a: a.HasProp("_GasteigerCharge")),
    ("MakeNodeFeaturesCall", None),  # will attempt make_node_features(atom)
]

# Storage
rows = []
missing_counter = Counter()
invalid_smiles = 0
valid_smiles = 0
full_ok = 0
examples_by_status = defaultdict(list)

for smi in tqdm(smiles_list, desc="checking SMILES"):
    rec = {"SMILES": smi, "valid": False, "num_atoms": 0, "missing": set(), "error": None}
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        invalid_smiles += 1
        rec["error"] = "RDKit failed to parse SMILES"
        rows.append(rec)
        examples_by_status["invalid"].append(smi)
        continue
    valid_smiles += 1
    # optional gasteiger
    if compute_gasteiger:
        try:
            rdPartialCharges.ComputeGasteigerCharges(mol, throwOnParamFailure=False)
        except Exception:
            # 计算可能失败，继续检查已有属性
            pass

    atoms_missing = set()
    atom_details = []
    ok_all_atoms = True
    for atom in mol.GetAtoms():
        atom_missing = []
        for name, fn in atom_checks:
            if name == "MakeNodeFeaturesCall":
                # try calling make_node_features from data.py (if imported)
                try:
                    feats = make_node_features(atom)
                except Exception as e:
                    atom_missing.append("MakeNodeFeaturesCall")
                continue
            try:
                val = fn(atom)
                # treat None as missing (some methods may return None)
                if val is None:
                    atom_missing.append(name)
            except Exception:
                atom_missing.append(name)
        if atom_missing:
            ok_all_atoms = False
            atoms_missing.update(atom_missing)
        atom_details.append({"idx": atom.GetIdx(), "symbol": atom.GetSymbol(), "missing": atom_missing})
    rec["valid"] = True
    rec["num_atoms"] = mol.GetNumAtoms()
    rec["missing"] = sorted(list(atoms_missing))
    if not atoms_missing:
        full_ok += 1
        examples_by_status["full_ok"].append(smi)
    else:
        examples_by_status["partial_missing"].append(smi)
        for m in atoms_missing:
            missing_counter[m] += 1
    rows.append(rec)

# Summary prints
total = len(smiles_list)
print(f"\nTOTAL SMILES: {total}")
print(f"Valid mols: {valid_smiles}")
print(f"Invalid SMILES: {invalid_smiles}")
print(f"SMILES with all atom checks OK: {full_ok}")
print("\nMissing counts (per-field):")
for k, v in missing_counter.most_common():
    print(f"  {k}: {v}")

# Show examples
def show_examples(key, n=5):
    lst = examples_by_status.get(key, [])
    print(f"\nExamples for '{key}' (showing up to {n}): total {len(lst)}")
    for s in lst[:n]:
        print("  ", s)

show_examples("invalid", max_examples_print)
show_examples("full_ok", max_examples_print)
show_examples("partial_missing", max_examples_print)

# Convenience: function to dump atom-level details for selected SMILES
def inspect_smiles(smi: str, compute_gasteiger_local: bool = True):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        print("Invalid SMILES:", smi); return
    if compute_gasteiger_local:
        try:
            rdPartialCharges.ComputeGasteigerCharges(mol, throwOnParamFailure=False)
        except Exception:
            pass
    print(f"SMILES: {smi}  | atoms: {mol.GetNumAtoms()}")
    for atom in mol.GetAtoms():
        idx = atom.GetIdx()
        sym = atom.GetSymbol()
        props = {}
        try:
            props["mass"] = atom.GetMass()
            props["atom_map"] = atom.GetAtomMapNum()
            props["is_aromatic"] = bool(atom.GetIsAromatic())
            props["formal_charge"] = atom.GetFormalCharge()
            props["Z"] = atom.GetAtomicNum()
            props["chiral_tag"] = int(atom.GetChiralTag())
            props["hybridization"] = atom.GetHybridization()
            props["degree"] = atom.GetDegree()
            props["total_Hs"] = atom.GetTotalNumHs()
            props["is_in_ring"] = atom.IsInRing()
            props["_GasteigerCharge_present"] = atom.HasProp("_GasteigerCharge")
            if atom.HasProp("_GasteigerCharge"):
                try:
                    props["_GasteigerCharge"] = float(atom.GetProp("_GasteigerCharge"))
                except Exception:
                    props["_GasteigerCharge"] = atom.GetProp("_GasteigerCharge")
        except Exception as e:
            props["error"] = str(e)
        # try make_node_features
        try:
            feats = make_node_features(atom)
            props["make_node_features_len"] = len(feats)
        except Exception as e:
            props["make_node_features_error"] = str(e)
        print(f"  idx={idx} {sym} -> {props}")

# Usage examples:
# - 查看第一个部分缺失的样本
if examples_by_status["partial_missing"]:
    print("\nInspecting one partial-missing example:")
    inspect_smiles(examples_by_status["partial_missing"][0])
else:
    print("\nNo partial-missing examples found.")
# - 如需自选 SMILES，调用 inspect_smiles("CCO")

Using train_filtered with 10060 SMILES


checking SMILES: 100%|██████████| 10060/10060 [00:07<00:00, 1376.46it/s]


TOTAL SMILES: 10060
Valid mols: 10060
Invalid SMILES: 0
SMILES with all atom checks OK: 10060

Missing counts (per-field):

Examples for 'invalid' (showing up to 8): total 0

Examples for 'full_ok' (showing up to 8): total 10060
   *.*C/C=C/CC.*CC(*)C#N.*CCC(C*)C(=O)O
   */C(=C(/*)c1ccccc1)c1ccccc1
   */C(=C(\c1ccccc1)c1ccc(*)cc1)c1ccccc1
   */C(F)=C(\F)C(F)(C(*)(F)F)C(F)(F)F
   */C=C(/*)C#CCCCCCCCCCCCCCCCCCCCCC(=O)O
   */C=C(/*)CCCCCCCCCCCCCCCCCCCCC(=O)O
   */C=C(\C#N)C(=O)Nc1cccc(NC(=O)/C(C#N)=C/c2ccc(/C=C/c3ccc(N(c4ccccc4)c4ccc(N(c5ccccc5)c5ccc(/C=C/c6ccc(*)s6)cc5)cc4)cc3)s2)c1
   */C=C/C(C(=O)OC(C)C)C(*)C(=O)OC(C)C

Examples for 'partial_missing' (showing up to 8): total 0

No partial-missing examples found.
